In [ ]:
from google.colab import files
uploaded = files.upload()
import os, zipfile
zip_name = next((n for n in uploaded if n.lower().endswith('.zip')), None)
if zip_name is None: raise FileNotFoundError('Upload Parking_Utilization_Datasets_FINAL.zip')
WORK='/content/parking_utilization'; os.makedirs(WORK,exist_ok=True)
with zipfile.ZipFile(zip_name,'r') as z: z.extractall(WORK)
print('Extracted:',WORK)

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
D=Path(WORK)
parking=pd.read_csv(D/'parking_sessions_raw.csv',parse_dates=['entry','exit'])
zones=pd.read_csv(D/'zones.csv'); payments=pd.read_csv(D/'payments.csv')
calendar=pd.read_csv(D/'calendar.csv',parse_dates=['date']); weather=pd.read_csv(D/'weather.csv',parse_dates=['date'])
calendar['date']=calendar['date'].dt.normalize(); weather['date']=weather['date'].dt.normalize()
print(parking.shape,zones.shape,payments.shape,calendar.shape,weather.shape)

In [ ]:
print('Missing values:'); display(parking.isna().sum())
print('Exact duplicates:',parking.duplicated().sum())
print('Negative durations:',(parking.duration<0).sum())
zone_map={'Zone 1':'Z001','z2':'Z002','zone_3':'Z003','Z-4':'Z004','zone5':'Z005'}
clean=parking.copy(); clean['zone']=clean.zone.replace(zone_map)
clean=clean.drop_duplicates(['zone','entry','exit']); clean=clean[clean.duration>=0].copy()
clean['duration']=clean.duration.fillna(clean.groupby('zone').duration.transform('median'))
clean['exit']=clean['exit'].fillna(clean.entry+pd.to_timedelta(clean.duration,unit='h'))
q1,q3=clean.duration.quantile([.25,.75]); upper=q3+1.5*(q3-q1)
clean['duration']=clean.duration.clip(upper=upper); clean['exit']=clean.entry+pd.to_timedelta(clean.duration,unit='h')
clean['date']=clean.entry.dt.normalize(); clean['hour']=clean.entry.dt.hour
print('Clean sessions:',len(clean))

In [ ]:
active=[]
for zone,g in clean.groupby('zone'):
    for s,e in zip(g.entry,g.exit):
        active.extend((zone,t) for t in pd.date_range(s.floor('h'),e.floor('h'),freq='h'))
active=pd.DataFrame(active,columns=['zone','datetime']); active['date']=active.datetime.dt.normalize(); active['hour']=active.datetime.dt.hour
analysis=(active.groupby(['zone','date','hour']).size().reset_index(name='active_sessions').merge(zones[['zone','location','capacity','zone_type']],on='zone'))
analysis['occupancy_pct']=analysis.active_sessions/analysis.capacity*100
analysis['occupancy_class']=pd.cut(analysis.occupancy_pct,[-np.inf,30,60,np.inf],labels=['Low','Medium','High'])
starts=clean.groupby(['zone','date','hour']).agg(session_starts=('session_id','count'),avg_duration=('duration','mean')).reset_index()
analysis=(analysis.merge(starts,on=['zone','date','hour'],how='left').merge(calendar,on='date',how='left').merge(weather,on='date',how='left'))
analysis['holiday']=analysis.holiday.fillna(False); analysis['event']=analysis.event.fillna(False); analysis['rain']=analysis.rain.fillna(False); analysis['temperature']=analysis.temperature.fillna(weather.temperature.median())
analysis['dayofweek_num']=analysis.date.dt.dayofweek; analysis['is_holiday']=analysis.holiday.astype(int); analysis['is_event']=analysis.event.astype(int); analysis['is_raining']=analysis.rain.astype(int)
zone_summary=analysis.groupby(['zone','zone_type'],as_index=False).agg(avg_occupancy=('occupancy_pct','mean'),max_occupancy=('occupancy_pct','max'),total_sessions=('session_starts','sum')).sort_values('avg_occupancy',ascending=False)
hour_summary=analysis.groupby('hour',as_index=False).agg(avg_occupancy=('occupancy_pct','mean'),total_sessions=('session_starts','sum')).sort_values('avg_occupancy',ascending=False)
turnover=clean.groupby('zone').agg(total_sessions=('session_id','count'),days_observed=('date','nunique')).reset_index().merge(zones[['zone','capacity','zone_type']],on='zone')
turnover['avg_daily_turnover']=turnover.total_sessions/(turnover.days_observed*turnover.capacity)
event_summary=analysis.groupby('event',as_index=False).agg(avg_occupancy=('occupancy_pct','mean'),avg_duration=('avg_duration','mean'))
rain_summary=analysis.groupby('rain',as_index=False).agg(avg_occupancy=('occupancy_pct','mean'),avg_duration=('avg_duration','mean'))
duration_summary=clean.merge(zones[['zone','zone_type']],on='zone').groupby('zone_type',as_index=False).agg(avg_duration=('duration','mean'),median_duration=('duration','median'))
print('Q1 Highest occupancy zones'); display(zone_summary.head(10))
print('Q2 Peak periods'); display(hour_summary.head(10))
print('Q3 Low turnover'); display(turnover.sort_values('avg_daily_turnover').head(10))
print('Q4 Event effect'); display(event_summary)
print('Q5 Rain effect'); display(rain_summary)
print('Q6 Duration by zone type'); display(duration_summary)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9,5)); z=zone_summary.head(10).sort_values('avg_occupancy'); plt.barh(z.zone,z.avg_occupancy); plt.xlabel('Average occupancy (%)'); plt.title('Top 10 Zones'); plt.show()
plt.figure(figsize=(9,5)); hs=hour_summary.sort_values('hour'); plt.plot(hs.hour,hs.avg_occupancy,marker='o'); plt.xlabel('Hour'); plt.ylabel('Occupancy (%)'); plt.title('Peak-Time Analysis'); plt.show()
plt.figure(figsize=(9,5)); plt.hist(clean.duration,bins=30); plt.xlabel('Duration (hours)'); plt.ylabel('Sessions'); plt.title('Duration Distribution'); plt.show()

In [ ]:
features=['zone','zone_type','capacity','hour','dayofweek_num','is_holiday','is_event','is_raining','temperature','session_starts','avg_duration']
X=analysis[features]; y=analysis.occupancy_class.astype(str)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix
pre=ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),['zone','zone_type']),('num','passthrough',[c for c in features if c not in ['zone','zone_type']])])
pipe=Pipeline([('pre',pre),('model',RandomForestClassifier(n_estimators=160,max_depth=16,random_state=42,n_jobs=-1,class_weight='balanced_subsample'))])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y); pipe.fit(Xtr,ytr); pred=pipe.predict(Xte)
print('Accuracy:',accuracy_score(yte,pred)); print('Precision:',precision_score(yte,pred,average='weighted')); print('Recall:',recall_score(yte,pred,average='weighted')); print('F1:',f1_score(yte,pred,average='weighted')); print('Confusion matrix:\n',confusion_matrix(yte,pred,labels=['Low','Medium','High']))
names=pipe.named_steps['pre'].get_feature_names_out(); imp=pipe.named_steps['model'].feature_importances_
fi=pd.DataFrame({'encoded_feature':names,'importance':imp})
def grp(s):
    s=s.replace('num__','').replace('cat__','')
    if s.startswith('zone_type_'): return 'zone_type'
    if s.startswith('zone_'): return 'zone'
    return s
fi['original_feature']=fi.encoded_feature.map(grp)
selected=fi.groupby('original_feature',as_index=False).importance.sum().sort_values('importance',ascending=False)
selected['selected']=selected.importance>0.02
display(selected)

In [ ]:
analysis.to_csv('/content/final_hourly_analysis.csv',index=False)
clean.to_csv('/content/parking_sessions_cleaned.csv',index=False)
selected.to_csv('/content/selected_features.csv',index=False)
print('Evidence CSVs created in /content')

In [ ]:

print(sorted(sessions['zone'].astype(str).unique())[:30])


In [ ]:

print(zones['zone_type'].value_counts(dropna=False))


In [ ]:

print(payments['payment_mode'].value_counts(dropna=False))


In [ ]:

print(payments['amount'].describe())


In [ ]:

print(sessions['duration'].describe())


In [ ]:

print(sessions['fee'].describe())


In [ ]:

hourly_starts=analysis.groupby('hour').size().reset_index(name='session_starts'); display(hourly_starts)


In [ ]:
# Extra 8 — weekday distribution
print(calendar['weekday'].value_counts(dropna=False))


In [ ]:

print(calendar['holiday'].value_counts(dropna=False))


In [ ]:

print(calendar['event'].fillna('None').value_counts().head(15))


In [ ]:
print(weather[['rain','temperature']].describe())


In [ ]:
print((hourly['occupancy_class'].value_counts(normalize=True)*100).round(2))


In [ ]:
zone_type_analysis=hourly.groupby('zone_type',as_index=False)['occupancy_pct'].mean().sort_values('occupancy_pct',ascending=False); display(zone_type_analysis)


In [ ]:

band_analysis=analysis.groupby('time_band',observed=False).size().reset_index(name='sessions'); display(band_analysis)


In [ ]:

print(analysis.groupby('zone_type')['duration'].median().sort_values(ascending=False))


In [ ]:

print(analysis['zone'].value_counts().head(10))


In [ ]:

num_cols=['duration','fee','hour']; display(sessions[num_cols].corr())


In [ ]:

percentiles=hourly['occupancy_pct'].quantile([.25,.5,.75,.9,.95]).rename('value').to_frame(); display(percentiles)


In [ ]:

print(zones.nlargest(10,'capacity')[['zone','capacity','zone_type']])


In [ ]:

print(zones.nsmallest(10,'capacity')[['zone','capacity','zone_type']])


In [ ]:

event_diff=event_comparison.set_index('day_type')['avg_occupancy_pct']; print(event_diff)


In [ ]:

weather_diff=weather_comparison.set_index('weather_condition')['avg_occupancy_pct']; print(weather_diff)


In [ ]:

fee_payment=payments.groupby('payment_mode')['amount'].mean().sort_values(ascending=False); display(fee_payment)


In [ ]:

payment_check=payments.groupby('session_id')['amount'].sum().describe(); print(payment_check)


In [ ]:

print('Unmatched zone rows:',analysis['capacity'].isna().sum())


In [ ]:
print('Unmatched calendar rows:',analysis['event'].isna().sum())


In [ ]:

print('Unmatched weather rows:',analysis['rain'].isna().sum())


In [ ]:

display(ml_model.isna().sum().to_frame('missing'))


In [ ]:

display(y.value_counts().to_frame('count'))


In [ ]:

display(pd.DataFrame({'actual':y_test.values[:20],'predicted':y_pred[:20]}))


In [ ]:

from sklearn.metrics import precision_recall_fscore_support
pr,rc,f1,sup=precision_recall_fscore_support(y_test,y_pred,labels=['Low','Medium','High']); display(pd.DataFrame({'class':['Low','Medium','High'],'precision':pr,'recall':rc,'f1':f1,'support':sup}))


In [ ]:

top_imp=importance_df.head(10).sort_values('importance'); plt.figure(figsize=(8,5)); plt.barh(top_imp.feature,top_imp.importance); plt.title('Top Feature Importance'); plt.xlabel('Permutation importance'); plt.tight_layout(); plt.show()


In [ ]:

pd.DataFrame({'selected_feature':selected_features}).to_csv(OUTPUT/'selected_features.csv',index=False)


In [ ]:

hourly[['zone','hour','occupancy_pct','occupancy_class']].to_csv(OUTPUT/'occupancy_classification.csv',index=False)


In [ ]:

zone_type_analysis.to_csv(OUTPUT/'zone_type_occupancy.csv',index=False)


In [ ]:

hourly_starts.to_csv(OUTPUT/'hourly_session_starts.csv',index=False)


In [ ]:

for f in sorted(OUTPUT.iterdir()): print(f.name, f.stat().st_size, 'bytes')


In [ ]:

summary=pd.DataFrame({'dataset':['sessions_cleaned','hourly_occupancy','ml_dataset'],'rows':[len(sessions),len(hourly),len(ml_model)]}); display(summary)


In [ ]:

print('Highest zone:',zone_analysis.iloc[0].zone); print('Peak hour:',int(peak_time.iloc[0].hour)); print('Lowest turnover:',turnover.iloc[0].zone)


In [ ]:

RANDOM_STATE=42; print('Random state used:',RANDOM_STATE)
